# Evaluate Gate checkpoints trên AdVQA (Kaggle)

Notebook clone SelTDA, đọc bốn student checkpoint, AdVQA validation và ảnh từ Kaggle dataset, rồi chấm bằng VQA accuracy chính thức.

Trước khi chạy, attach COCO Caption 2014 (chulij/cocodatasets) và private dataset phong2004/seltda-gate-checkpoints. Notebook chỉ đọc checkpoint từ Kaggle Dataset, không tải từ Google Drive. Output nằm trong /kaggle/working/advqa_evals.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/fantastichaha11/SelTDA.git'
BRANCH = 'feat/pseudo-label-filter'
REPO_DIR = Path('/kaggle/working/SelTDA')
DATA_ROOT = Path('/kaggle/working/data')
ADVQA_ROOT = DATA_ROOT / 'advqa'
COCO_ROOT = Path('/kaggle/input/cocodatasets/COCO')
CHECKPOINT_INPUT_DIR = Path('/kaggle/input/seltda-gate-checkpoints')
OUTPUT_ROOT = Path('/kaggle/working/advqa_evals')
CONFIG_PATH = REPO_DIR / 'configs/advqa_eval_kaggle.yaml'

CHECKPOINT_FILES = {
    'gate_1': 'checkpoint_09_g1.pth',
    'gate_2': 'checkpoint_09_g2.pth',
    'gate_1_2': 'checkpoint_09_g12.pth',
    'gate_4': 'checkpoint_09_g4.pth',
}

MAX_GPUS = 2
BATCH_SIZE_TEST = 8
K_TEST = 128
MAX_RECORDS = None  # Ví dụ 128 để smoke test; None để eval đủ 10.000 câu

In [ ]:
import subprocess, sys

if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)])
else:
    print('Repo đã tồn tại, bỏ qua clone:', REPO_DIR)

# Cài dependency tối thiểu, không pin hoặc force-reinstall NumPy/SciPy của Kaggle.
packages = [
    'omegaconf==2.3.0',
    'hydra-core==1.3.2',
    'timm==0.4.12',
    'fairscale==0.4.13',
    'transformers==4.36.1',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])
print('Clone và dependency setup hoàn tất.')

In [ ]:
if not CHECKPOINT_INPUT_DIR.is_dir():
    raise FileNotFoundError(
        f'Không thấy {CHECKPOINT_INPUT_DIR}. Hãy Add Input -> '        'phong2004/seltda-gate-checkpoints trước khi chạy notebook.'
    )

checkpoint_paths = {}
for label, filename in CHECKPOINT_FILES.items():
    checkpoint = CHECKPOINT_INPUT_DIR / filename
    if not checkpoint.is_file() or checkpoint.stat().st_size < 1024 * 1024:
        raise RuntimeError(f'Checkpoint không hợp lệ: {label} -> {checkpoint}')
    checkpoint_paths[label] = checkpoint
    print(f'{label}: {checkpoint} ({checkpoint.stat().st_size / 1024**3:.2f} GiB)')


In [ ]:
import json, os, time
import requests
from tqdm.auto import tqdm

ADVQA_ROOT.mkdir(parents=True, exist_ok=True)
COCO_VAL_ROOT = COCO_ROOT / 'val2014'
assert COCO_VAL_ROOT.is_dir(), (
    f'Không thấy {COCO_VAL_ROOT}. Hãy attach Kaggle dataset chulij/cocodatasets.'
)

QUESTIONS_URL = 'https://dl.fbaipublicfiles.com/advqa/v1_OpenEnded_mscoco_val2017_advqa_questions.json'
ANNOTATIONS_URL = 'https://dl.fbaipublicfiles.com/advqa/v1_mscoco_val2017_advqa_annotations.json'
QUESTIONS_PATH = ADVQA_ROOT / 'v1_OpenEnded_mscoco_val2017_advqa_questions.json'
ANNOTATIONS_PATH = ADVQA_ROOT / 'v1_mscoco_val2017_advqa_annotations.json'

def download_file(url, destination):
    if destination.is_file() and destination.stat().st_size > 0:
        return
    tmp = destination.with_suffix(destination.suffix + '.tmp')
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with tmp.open('wb') as f:
            for chunk in response.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)
    os.replace(tmp, destination)

download_file(QUESTIONS_URL, QUESTIONS_PATH)
download_file(ANNOTATIONS_URL, ANNOTATIONS_PATH)
with QUESTIONS_PATH.open() as f:
    questions_payload = json.load(f)
with ANNOTATIONS_PATH.open() as f:
    annotations_payload = json.load(f)
raw_questions = questions_payload['questions']
raw_annotations = annotations_payload['annotations']
print(f'Questions={len(raw_questions):,}; annotations={len(raw_annotations):,}')

In [ ]:
annotation_by_qid = {int(a['question_id']): a for a in raw_annotations}
questions = [q for q in raw_questions if int(q['question_id']) in annotation_by_qid]
if MAX_RECORDS is not None:
    questions = questions[:MAX_RECORDS]

image_ids = sorted({int(q['image_id']) for q in questions})

def coco_filename(image_id):
    return f'COCO_val2014_{image_id:012d}.jpg'

missing_paths = [
    COCO_VAL_ROOT / coco_filename(image_id)
    for image_id in image_ids
    if not (COCO_VAL_ROOT / coco_filename(image_id)).is_file()
]
if missing_paths:
    preview = [str(path) for path in missing_paths[:10]]
    raise FileNotFoundError(
        f'Kaggle COCO dataset thiếu {len(missing_paths)} ảnh AdVQA. Ví dụ: {preview}'
    )
print(f'Đã xác minh {len(image_ids):,} ảnh AdVQA trong {COCO_VAL_ROOT}')


In [ ]:
records = []
for question in questions:
    qid = int(question['question_id'])
    annotation = annotation_by_qid[qid]
    answers = [item['answer'].strip() for item in annotation['answers'] if item.get('answer', '').strip()]
    records.append({
        'dataset': 'advqa',
        'image': f'val2014/{coco_filename(int(question["image_id"]))}',
        'question': question['question'].strip(),
        'question_id': qid,
        'answer': answers,
    })
answer_list = sorted({answer for record in records for answer in record['answer']})

def write_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False)
    os.replace(tmp, path)

# GenericVqaDataset luôn dựng cả train loader, kể cả --evaluate.
write_json(ADVQA_ROOT / 'train.json', records)
write_json(ADVQA_ROOT / 'val.json', records)
write_json(ADVQA_ROOT / 'answer_list.json', answer_list)
assert all((COCO_ROOT / record['image']).is_file() for record in records)
print(f'Eval records={len(records):,}; answer candidates={len(answer_list):,}')

import yaml
config = {
    'vqa_root': str(COCO_ROOT),
    'train_files': ['train'],
    'val_file': 'val',
    'ann_root': str(ADVQA_ROOT),
    'dataset_name': 'generic_vqa',
    'answer_list': 'answer_list',
    'truncate_train_dataset_to': None,
    'pretrained': str(next(iter(checkpoint_paths.values()))),
    'vit': 'base',
    'batch_size_train': 1,
    'batch_size_test': BATCH_SIZE_TEST,
    'vit_grad_ckpt': False,
    'vit_ckpt_layer': 0,
    'init_lr': 2e-5,
    'image_size': 480,
    'k_test': K_TEST,
    'inference': 'rank',
    'weight_decay': 0.05,
    'min_lr': 0,
    'max_epoch': 1,
    'torch_home': '/kaggle/working/torch_home',
    'wandb': False,
    'save_last_only': True,
}
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False))
print(CONFIG_PATH.read_text())

In [ ]:
import torch

assert torch.cuda.is_available(), 'Bật GPU trong Kaggle Settings > Accelerator'
gpu_count = min(MAX_GPUS, torch.cuda.device_count())
assert gpu_count == MAX_GPUS, f'Notebook cần {MAX_GPUS} GPU, nhưng chỉ thấy {torch.cuda.device_count()}'
for gpu_id in range(gpu_count):
    print(f'cuda:{gpu_id}:', torch.cuda.get_device_name(gpu_id))

smoke = subprocess.run(
    [sys.executable, '-c', 'import train_vqa; print("train_vqa import OK")'],
    cwd=REPO_DIR, text=True, capture_output=True
)
print(smoke.stdout)
if smoke.returncode != 0:
    raise RuntimeError(smoke.stderr)

In [ ]:
def run_checkpoint(label, checkpoint, gpu_id):
    output_dir = OUTPUT_ROOT / label
    output_dir.mkdir(parents=True, exist_ok=True)
    log_path = output_dir / 'inference.log'
    command = [
        sys.executable, '-m', 'torch.distributed.run',
        '--standalone',
        f'--nproc_per_node={gpu_count}',
        'train_vqa.py',
        '--config=configs/advqa_eval_kaggle.yaml',
        f'--output_dir={output_dir}',
        '--evaluate',
        '--no-resume',
        '--overrides',
        f'pretrained={checkpoint}',
    ]
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = ','.join(str(i) for i in range(gpu_count))
    env['OMP_NUM_THREADS'] = '2'
    with log_path.open('w') as log:
        completed = subprocess.run(command, cwd=REPO_DIR, env=env, stdout=log, stderr=subprocess.STDOUT)
    if completed.returncode != 0:
        tail = '\n'.join(log_path.read_text(errors='replace').splitlines()[-80:])
        raise RuntimeError(f'{label} failed on GPU {gpu_id}:\n{tail}')
    result_path = output_dir / 'result/vqa_result.json'
    if not result_path.is_file():
        raise RuntimeError(f'{label}: thiếu result {result_path}')
    return result_path

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
result_paths = {}
visible_gpus = ','.join(str(i) for i in range(gpu_count))
for label, checkpoint in checkpoint_paths.items():
    print(f'Bắt đầu {label} bằng DDP trên {gpu_count} GPU...')
    result_paths[label] = run_checkpoint(label, checkpoint, visible_gpus)
    print(label, '->', result_paths[label])
print('Inference hoàn tất.')

In [ ]:
import csv

# advqa_eval.py của repo hardcode path máy /net/acadia; đổi sang path Kaggle runtime.
eval_script = REPO_DIR / 'advqa_eval.py'
eval_source = eval_script.read_text()
MODIFIED_ANNOTATIONS_PATH = ADVQA_ROOT / 'val_annotations_with_question_type.json'
path_replacements = {
    '/net/acadia10a/data/zkhan/advqa/v1_mscoco_val2017_advqa_annotations.json': str(ANNOTATIONS_PATH),
    '/net/acadia10a/data/zkhan/advqa/v1_OpenEnded_mscoco_val2017_advqa_questions.json': str(QUESTIONS_PATH),
    '/net/acadia10a/data/zkhan/advqa/nb017_val2017_annotations_w_qtype.json': str(MODIFIED_ANNOTATIONS_PATH),
}
for old_path, new_path in path_replacements.items():
    if old_path not in eval_source:
        raise RuntimeError(f'advqa_eval.py không còn chứa path dự kiến: {old_path}')
    eval_source = eval_source.replace(old_path, new_path)
eval_script.write_text(eval_source)

summary = {}
for label in CHECKPOINT_FILES:
    result_path = result_paths[label]
    command = [sys.executable, 'advqa_eval.py', str(result_path)]
    completed = subprocess.run(
        command,
        cwd=REPO_DIR,
        text=True,
        capture_output=True,
    )
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f'advqa_eval.py failed cho {label}:\n{completed.stderr}')

    metric_path = result_path.parent / 'advqa_eval.json'
    with metric_path.open() as f:
        accuracy = json.load(f)
    summary[label] = accuracy
    print(label, 'overall =', accuracy['overall'])

SUMMARY_JSON = OUTPUT_ROOT / 'advqa_eval_summary.json'
write_json(SUMMARY_JSON, summary)

SUMMARY_CSV = OUTPUT_ROOT / 'advqa_eval_summary.csv'
answer_types = sorted({key for value in summary.values() for key in value.get('perAnswerType', {})})
with SUMMARY_CSV.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['checkpoint', 'overall', *answer_types])
    writer.writeheader()
    for label, accuracy in summary.items():
        row = {'checkpoint': label, 'overall': accuracy['overall']}
        row.update(accuracy.get('perAnswerType', {}))
        writer.writerow(row)

print('Summary JSON:', SUMMARY_JSON)
print('Summary CSV:', SUMMARY_CSV)
for label, accuracy in sorted(summary.items(), key=lambda item: item[1]['overall'], reverse=True):
    print(f'{label:10s} overall={accuracy["overall"]:.2f} perAnswerType={accuracy.get("perAnswerType", {})}')
